# **NLP Assignment 4 – BERT Fine-Tuning**

**1. Install Required Libraries**

In [ ]:
!pip install transformers datasets scikit-learn torch

**2. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
from torch.utils.data import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

**3. Load Dataset**

In [ ]:
import json

data = []

with open('/content/News_Category_Dataset_v3.json', 'r') as f:
    for i, line in enumerate(f):
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Skipping malformed line {i+1}: {e}")

df = pd.DataFrame(data)

df.head()

**4. Data Preprocessing**

In [ ]:
# Re-initialize df from original data to ensure 'headline' and 'short_description' columns are present
df = pd.DataFrame(data)

# Combine headline + short description
df['text'] = df['headline'] + " " + df['short_description']

# Drop missing values
df = df[['text', 'category']].dropna()

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

df.head()

**5. Train / Validation / Test Split**

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

**6. Tokenization (BERT)**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

**7. Create PyTorch Dataset**

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

**8. Load Pre-trained BERT Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(df['label'].unique())
)

**9. Training Argument**

In [ ]:
!pip uninstall -y transformers accelerate -q
!pip install transformers accelerate -q

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50
)

**10. Evaluation Metrics**

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

**11. Trainer Setup**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

**12. Train Model**

In [ ]:
trainer.train()

**13. Evaluate Model**

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

**14. Confusion Matrix**

In [ ]:
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)

cm = confusion_matrix(test_labels, y_pred)
print(cm)

**15. Confusion Matrix**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# **EXPERIMENTS SECTION**

**Experiment 1: Freeze BERT Layers**

In [ ]:
# Freeze DistilBERT layers
for param in model.distilbert.parameters():
    param.requires_grad = False

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
result_frozen = trainer.evaluate()
print(result_frozen)

**Experiment 2: Fine-tune last 2 layers**

In [ ]:
# Freeze all layers first
for param in model.distilbert.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for name, param in model.distilbert.named_parameters():
    if "transformer.layer.4" in name or "transformer.layer.5" in name:
        param.requires_grad = True

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
result_frozen = trainer.evaluate()
print(result_frozen)

# **FINAL COMPARISON TABLE**

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Experiment": [
        "Frozen DistilBERT",
        "Last 2 Layers Fine-Tuned",
        "Full Fine-Tuning"
    ],
    "Accuracy": [
        0.6739,
        0.6744,
        0.6716
    ],
    "F1 Score": [
        0.6695,
        0.6685,
        0.6614
    ],
    "Precision": [
        0.6686,
        0.6675,
        0.6621
    ],
    "Recall": [
        0.6739,
        0.6744,
        0.6716
    ]
})

results

**Graph: Accuracy and F1 Score Comparison**

In [ ]:
import matplotlib.pyplot as plt

experiments = ["Frozen", "Last 2 Layers", "Full Fine-Tuning"]
accuracy = [0.6739, 0.6744, 0.6716]
f1_scores = [0.6695, 0.6685, 0.6614]

plt.figure()

plt.plot(experiments, accuracy, marker='o', label="Accuracy")
plt.plot(experiments, f1_scores, marker='o', label="F1 Score")

plt.ylim(0.65, 0.68)

plt.title("Model Performance Comparison (Zoomed)")
plt.xlabel("Experiment Type")
plt.ylabel("Score")

plt.legend()
plt.grid()

plt.show()